# Chạy repo cuối trên GPU (RTX 4060)

Kernel: **Python (san pham GPU – kiem_repo_asr)** (= `kiem_repo_asr\.venv`, torch CUDA). Chọn ở góc trên bên phải.
Chạy từng ô bằng **Shift + Enter**, theo thứ tự. Ô nặng có ghi thời gian. Muốn dừng ô đang chạy: nút ■ (Interrupt) trên thanh notebook.

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Thiết lập + kiểm GPU | vài giây |
| 2 | Unit test (backend giả) | ~1 phút |
| 3 | Đánh giá cũ vs mới, 50 file (chạy thử) | ~5 phút |
| 4 | Đánh giá cũ vs mới, **857 file** | ~1–1,5 giờ |
| 5 | Đọc bảng kết quả mới nhất | vài giây |
| 6 | Nghiệm thu model thật (3 file thay thế) | ~5 phút |
| 7 | Demo Gradio (mở http://localhost:7860) | ~1 phút nạp model |

## 1. Thiết lập và kiểm GPU

In [1]:
import os, sys, subprocess, glob
from pathlib import Path

GOC = Path(r"D:\CAIDCM\teamwork\repo_cuoi")
os.chdir(GOC)
ENV = dict(os.environ,
           ASR_BACKEND="that",
           HF_HUB_CACHE=r"D:\CAIDCM\teamwork\task3\hf_home\hub",
           HF_HUB_OFFLINE="1",
           PYTHONIOENCODING="utf-8",
           PYTHONDONTWRITEBYTECODE="1",
           FLEURS_AUDIO_DIR=r"D:\CAIDCM\teamwork\task3\data\clean\audio_16k\fleurs\test")

def chay(*lenh, env=None):
    """Chạy một lệnh Python trong venv này, in kết quả từng dòng ngay khi có."""
    p = subprocess.Popen([sys.executable, *lenh], cwd=GOC, env=env or ENV, text=True,
                         encoding="utf-8", errors="replace", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for dong in p.stdout:
        if "Warning" in dong or "warnings.warn" in dong:
            continue
        print(dong, end="")
    p.wait()
    print(f"\n--- mã thoát {p.returncode} ---")
    return p.returncode

import torch
print("Python :", sys.executable)
print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    tu_do, tong = torch.cuda.mem_get_info()
    print("GPU    :", torch.cuda.get_device_name(0), f"| VRAM trống {tu_do/2**30:.1f}/{tong/2**30:.1f} GiB")
else:
    print("KHÔNG thấy GPU – đổi kernel sang 'Python (san pham GPU – kiem_repo_asr)'")

Python : D:\CAIDCM\teamwork\kiem_repo_asr\.venv\Scripts\python.exe
torch  : 2.0.1+cu117 | CUDA: True
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU | VRAM trống 6.9/8.0 GiB


## Kiểm bản đang dùng
Sản phẩm phải dùng adapter V2. Ô này dừng notebook nếu không đúng.

In [2]:
import hashlib, json
c = json.loads((GOC / "configs/ban_chot.json").read_text(encoding="utf-8"))
sys.path.insert(0, str(GOC))
from src.pipeline import ADAPTER_MAC_DINH
f = ADAPTER_MAC_DINH / "adapter_model.safetensors"
h = hashlib.sha256(f.read_bytes()).hexdigest()
d = [l.split()[0] for l in (GOC / "checkpoints/SHA256SUMS").read_text().splitlines() if l.split()[1] == "V2/best/adapter_model.safetensors"]
print("Bản chốt:", c["ban_chot"], "· adapter mặc định:", ADAPTER_MAC_DINH.relative_to(GOC).as_posix())
print("ADAPTER_PATH trong môi trường:", ENV.get("ADAPTER_PATH", "không đặt, dùng mặc định"))
print("SHA256 adapter:", h[:16], "· khớp SHA256SUMS:", h == d[0])
assert c["ban_chot"] == "V2" and ADAPTER_MAC_DINH.parts[-2:] == ("V2", "best") and "ADAPTER_PATH" not in ENV and h == d[0]
print("ĐẠT: mọi ô bên dưới dùng adapter V2")

Bản chốt: V2 · adapter mặc định: checkpoints/V2/best
ADAPTER_PATH trong môi trường: không đặt, dùng mặc định
SHA256 adapter: a83e1eaa8a42ad01 · khớp SHA256SUMS: True
ĐẠT: mọi ô bên dưới dùng adapter V2


## 2. Unit test (backend giả, không cần GPU)

In [3]:
chay("-m", "pytest", "-p", "no:cacheprovider", "-q", env=dict(ENV, ASR_BACKEND="gia"))

..............................................................s.sssss... [ 47%]


........................................................................ [ 95%]


.......                                                                  [100%]
============================== warnings summary ===============================
..\kiem_repo_asr\.venv\Lib\site-packages\starlette\testclient.py:40
    _PortalFactoryType = typing.Callable[[], typing.ContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html

--- mã thoát 0 ---


0

## 3. Đánh giá cách xử lý cũ vs mới – chạy thử 50 file (~5 phút)
So `truoc_0411` (cắt giữa, không bỏ lặp, không đổi số) với `proposed` (cắt tại chỗ lặng, bỏ lặp, đổi số) trên cùng tập.

In [4]:
chay("tests/danh_gia_proposed.py", "--tap", "test", "--n", "50")

  return self.fget.__get__(instance, owner)()


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


truoc_0411: 0/50


proposed: 0/50


# Danh gia Proposed Method · fleurs_test

`vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best`

| Cau hinh | Nhom | n | WER_asr | CER_asr | WER_sau_doi_so | BLEU | chrF++ | RTF |
|---|---|---|---|---|---|---|---|---|
| truoc_0411 | tat_ca | 50 | 8.98 | 6.69 | 8.98 | 28.15 | 54.25 | 0.1258 |
| proposed | tat_ca | 50 | 8.98 | 6.69 | 7.71 | 28.08 | 54.25 | 0.1332 |

WER_asr: loi PhoWhisper nguyen ban (so voi Task 2: FLEURS test 13,68 khi chay tung file).
WER_sau_doi_so: tham khao, tham chieu FLEURS viet chu so nen doi so lam WER giam.
BLEU/chrF++: sacreBLEU 2.4.3, cuoi pipeline (cascade).

Da ghi D:\CAIDCM\teamwork\repo_cuoi\danh_gia\20261105_131513



--- mã thoát 0 ---


0

## 4. Đánh giá đầy đủ – 857 file FLEURS test (~1–1,5 giờ)
Chạy ô 3 ổn rồi mới chạy ô này. Tắt game/ứng dụng nặng. Kết quả ghi vào `danh_gia/<ngày_giờ>/`.

In [5]:
chay("tests/danh_gia_proposed.py", "--tap", "test")

  return self.fget.__get__(instance, owner)()


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


truoc_0411: 0/857


truoc_0411: 100/857


truoc_0411: 200/857


truoc_0411: 300/857


... (13 outputs left out)

proposed: 800/857


# Danh gia Proposed Method · fleurs_test

`vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best`

| Cau hinh | Nhom | n | WER_asr | CER_asr | WER_sau_doi_so | BLEU | chrF++ | RTF |
|---|---|---|---|---|---|---|---|---|
| truoc_0411 | tat_ca | 857 | 12.27 | 8.6 | 12.27 | 28.01 | 53.03 | 0.1249 |
| truoc_0411 | dai | 10 | 16.03 | 12.04 | 16.03 | 15.51 | 49.61 | 0.1004 |
| proposed | tat_ca | 857 | 12.26 | 8.57 | 10.12 | 28.15 | 53.1 | 0.1192 |
| proposed | dai | 10 | 15.59 | 10.99 | 12.94 | 18.74 | 51.34 | 0.1082 |

WER_asr: loi PhoWhisper nguyen ban (so voi Task 2: FLEURS test 13,68 khi chay tung file).
WER_sau_doi_so: tham khao, tham chieu FLEURS viet chu so nen doi so lam WER giam.
BLEU/chrF++: sacreBLEU 2.4.3, cuoi pipeline (cascade).

Da ghi D:\CAIDCM\teamwork\repo_cuoi\danh_gia\20261105_131807



--- mã thoát 0 ---


0

## 5. Đọc bảng kết quả mới nhất

In [6]:
from IPython.display import Markdown, display
ds = sorted(glob.glob(str(GOC / "danh_gia" / "*" / "bang.md")))
if ds:
    print("Đọc:", ds[-1])
    display(Markdown(Path(ds[-1]).read_text(encoding="utf-8")))
else:
    print("Chưa có kết quả – chạy ô 3 hoặc 4 trước")

Đọc: D:\CAIDCM\teamwork\repo_cuoi\danh_gia\20261105_131807\bang.md


# Danh gia Proposed Method · fleurs_test

`vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best`

| Cau hinh | Nhom | n | WER_asr | CER_asr | WER_sau_doi_so | BLEU | chrF++ | RTF |
|---|---|---|---|---|---|---|---|---|
| truoc_0411 | tat_ca | 857 | 12.27 | 8.6 | 12.27 | 28.01 | 53.03 | 0.1249 |
| truoc_0411 | dai | 10 | 16.03 | 12.04 | 16.03 | 15.51 | 49.61 | 0.1004 |
| proposed | tat_ca | 857 | 12.26 | 8.57 | 10.12 | 28.15 | 53.1 | 0.1192 |
| proposed | dai | 10 | 15.59 | 10.99 | 12.94 | 18.74 | 51.34 | 0.1082 |

WER_asr: loi PhoWhisper nguyen ban (so voi Task 2: FLEURS test 13,68 khi chay tung file).
WER_sau_doi_so: tham khao, tham chieu FLEURS viet chu so nen doi so lam WER giam.
BLEU/chrF++: sacreBLEU 2.4.3, cuoi pipeline (cascade).

## 6. Nghiệm thu model thật (`tests/nghiem_thu_that.py`)
Đang dùng 3 file FLEURS dev **thay thế**. Có audio tự thu thì sửa danh sách `AUDIO` (3–5 file wav 10–20 s).

In [7]:
AUDIO = [r"D:\CAIDCM\teamwork\kiem_repo_asr2\audio_thay_the\dev1_16525570830762756295.wav",
         r"D:\CAIDCM\teamwork\kiem_repo_asr2\audio_thay_the\dev2_13754775770768255557.wav",
         r"D:\CAIDCM\teamwork\kiem_repo_asr2\audio_thay_the\dev3_15730294784062856962.wav"]
chay("tests/nghiem_thu_that.py", "--audio", *AUDIO)

[DAT ] T07 Nap loi: FileNotFoundError: khong thay adapter khong_ton_tai\V9
nap model that (lan dau tai ~2,7 GB) ...


  return self.fget.__get__(instance, owner)()


da nap: vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[DAT ] T01/F01 Audio that: vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best
    dev1_16525570830762756295.wav: VI='các ví dụ về hoạt động tại chỗ lao gầm săn bắn câu cá chụp ảnh ngắm chim và tham' | EN='Examples of on-site activities include roaring, hunting, fishing, photography, b' | RTF 0.216
    dev2_13754775770768255557.wav: VI='nghiên cứu trí tuệ nhân tạo bao gồm tạo ra những cỗ máy tự động hóa các tác dụ đ' | EN='The study of artificial intelligence involves creating machines that automate ar' | RTF 0.136
    dev3_15730294784062856962.wav: VI='chi phí chiến tranh của chủ nghĩa đế quốc hoa kỳ trong cuộc trình phạt philippin' | EN='The cost of American imperialism in the war against the Philippines was paid by ' | RTF 0.128
    -> NGUOI KIEM phai doc lai: noi dung VI/EN co dung voi audio khong


[DAT ] T03/F03 Audio dai: 1 phut: 3 doan, dai nhat 20.8 s, lien nhau=True, phu het=True, doan rong=0, xu ly 6.4 s
    3 phut: 7 doan, dai nhat 26.0 s, lien nhau=True, phu het=True, doan rong=0, xu ly 18.7 s
    5 phut: 12 doan, dai nhat 27.8 s, lien nhau=True, phu het=True, doan rong=0, xu ly 28.6 s
    -> NGUOI KIEM nghe cho noi 2 doan: co lap/mat chu khong


... (2 outputs left out)

[DAT ] T08/F06 Doi phien ban: V2 mac dinh: vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best
      EN='Examples of on-site activities include roaring, hunting, fishing, photography, bird watchi'
    V4: vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V4/best
      EN='Examples of on-site activities: roaring, hunting, fishing, photography, bird watching and '
    B0: vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA khong (B0)
      EN='examples of on-site activities roaring hunting fishing birdwatching photography and park t'
    ban dich 3 ban giong nhau het: False (giong het thi nghi adapter khong duoc nap)



6/7 muc DAT. Bang: D:\CAIDCM\teamwork\repo_cuoi\nghiem_thu\20261105_140242\bang.md



--- mã thoát 1 ---


1

## 7. Demo Gradio (model thật, V2)
Ô này chạy demo **nền**; mở http://localhost:7860 trên Chrome (thử được micro). Dừng demo bằng ô 8.

In [8]:
demo = subprocess.Popen([sys.executable, "app/gradio_app.py"], cwd=GOC, env=dict(ENV, GRADIO_SHARE="0"),
                        stdout=open(GOC / "demo.log", "w", encoding="utf-8"), stderr=subprocess.STDOUT)
import time
for _ in range(60):
    time.sleep(2)
    if "7860" in (GOC / "demo.log").read_text(encoding="utf-8", errors="replace"):
        print("Demo đã chạy: http://localhost:7860"); break
else:
    print("Chưa thấy demo lên – xem demo.log")

Demo đã chạy: http://localhost:7860


In [9]:
dong = [l.strip() for l in (GOC / "demo.log").read_text(encoding="utf-8", errors="replace").splitlines() if "backend=" in l or "gan adapter" in l]
print("\n".join(x[:150] for x in dong))
assert any("V2/best" in x or "V2\\best" in x for x in dong) and not any("V4" in x for x in dong)
print("ĐẠT: demo đang chạy adapter V2")

INFO:src.pipeline:gan adapter LoRA D:\CAIDCM\teamwork\repo_cuoi\checkpoints\V2\best
INFO:__main__:che do tu_nap, backend=that, vinai/PhoWhisper-small@a86b604c + vinai/vinai-translate-vi2en-v2@ae7baa85 + LoRA V2/best
ĐẠT: demo đang chạy adapter V2


## 8. Dừng demo

In [10]:
try:
    demo.terminate(); demo.wait(10); print("Đã dừng demo")
except NameError:
    print("Demo chưa chạy")

Đã dừng demo
